In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim
from pyspark.sql.window import Window

In [0]:
df = spark.table('workspace.bronze.crm_prd_info')

In [0]:
df.display()

# Data transformation - TO DO list after exploring the data
* ~~trimming whitespaces for all string type columns~~
* ~~ROW #1 is the column names - to correct this from bronze, you need to add the header & inferschema in order to work. So not this no required~~
* ~~prd_key parsing - meaning split the prd_key and create another column called cat_id~~
* ~~Normalize prd_line data to a friendly names~~
    * these are the available abbreviation in the column ('M', 'S', 'T', 'R'), however, I'm not sure what these letters stands for, I may need to check cheatsheet
* ~~Rename header/column names to a much friendly names~~
    * prd_id - product_id
    * prd_key - product_key
    * prd_nm - product_name
    * prd_cost - product_cost
    * prd_line - product_line
    * prd_start_dt - start_date
    * prd_end_dt - end_date
* ~~column data type need to change~~
    * prd_id to int type
    * prd_start_dt & prd_end_dt to date type
* ~~there are NULL values~~
    * what are we doing with NULL values?
        * ~~prd_cost - we replaced ALL NULL values to 0~~
* ~~there are no empty values~~

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

df.display()

In [0]:
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

df.display()

In [0]:
df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))

df.display()

In [0]:
df = (
    df
    # Normalize product line
    .withColumn(
        "prd_line",
        F.when(F.upper(col("prd_line")) == "M", "Mountain")
         .when(F.upper(col("prd_line")) == "R", "Road")
         .when(F.upper(col("prd_line")) == "S", "Other Sales")
         .when(F.upper(col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)

df.display()

In [0]:
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))
# note: there is no changes to this as the type was alraedy a date
df.display()

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

df.display()

# Write into Silver Layer

In [0]:
(
  df.write
    .mode('overwrite')
    .format('delta')
    .saveAsTable('workspace.silver.crm_prd_info')
)

In [0]:
%sql
SELECT
*
FROM workspace.silver.crm_prd_info